# Notebook 05: FastAPI Web Service

本实验演示如何通过 REST API 与智能家居 Agent 交互。

## 前置条件
1. 在终端启动 API Server: `python api_server.py`
2. 确保 Neo4j 正在运行并已加载种子数据
3. 确保 `.env` 中配置了 LLM Provider

## 学习内容
- 用 httpx 调用 REST API
- 测试各种端点（健康检查、聊天、图谱查询）
- 消费 SSE 流式响应
- 测试限流中间件

In [ ]:
import httpx
import json
import time

# API 基础地址
BASE_URL = "http://localhost:8000"
client = httpx.Client(base_url=BASE_URL, timeout=60.0)

print(f"API Client ready: {BASE_URL}")

## 1. 健康检查

首先验证 API 是否正常运行。

In [ ]:
# 健康检查
r = client.get("/api/v1/health")
print(f"Status Code: {r.status_code}")
print(f"Response:")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

## 2. 图谱查询端点

这些端点直接查询 Neo4j 知识图谱，不需要 LLM。

In [ ]:
# 图谱统计
r = client.get("/api/v1/graph/stats")
print("=== 图谱统计 ===")
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

In [ ]:
# 列出所有房间
r = client.get("/api/v1/graph/rooms")
print("=== 所有房间 ===")
for room in r.json():
    print(f"  {room.get('name', 'N/A')} ({room.get('type', 'N/A')}, {room.get('floor', 'N/A')})")

In [ ]:
# 查看客厅的设备
r = client.get("/api/v1/graph/rooms/living/devices")
data = r.json()
print(f"=== {data.get('room', 'N/A')} 的设备 (共 {data.get('device_count', 0)} 个) ===")
print(json.dumps(data, indent=2, ensure_ascii=False)[:1000])

In [ ]:
# 列出所有场景
r = client.get("/api/v1/graph/scenes")
print("=== 所有场景 ===")
for scene in r.json():
    rooms = ', '.join(scene.get('rooms', []))
    print(f"  {scene.get('name', 'N/A')} (mood: {scene.get('mood', 'N/A')}) - {rooms}")

## 3. 聊天端点（同步）

向 Agent 发送消息，获取完整响应。

In [ ]:
# 发送聊天消息
r = client.post("/api/v1/chat", json={
    "message": "打开客厅的灯",
    "debug": True,
})

data = r.json()
print(f"=== Agent 回复 ===")
print(f"响应: {data['response']}")
print(f"\n耗时: {data['metadata'].get('duration_seconds', 'N/A')}s")

if data.get('reasoning_trace'):
    print(f"\n=== 推理过程 ===")
    for step in data['reasoning_trace']:
        print(f"  - {step}")

In [ ]:
# 尝试更复杂的请求
r = client.post("/api/v1/chat", json={
    "message": "我想看电影，帮我调整客厅的氛围",
    "debug": True,
})

data = r.json()
print(f"响应: {data['response']}")
print(f"\n推理步骤: {len(data.get('reasoning_trace', []))} 步")

## 4. SSE 流式响应

使用 httpx 的流式接口消费 Server-Sent Events。

每个事件对应 Agent 工作流中一个节点的完成。

In [ ]:
# SSE 流式请求
print("=== 流式响应 ===")
with httpx.stream(
    "POST",
    f"{BASE_URL}/api/v1/chat/stream",
    json={"message": "把卧室灯调暗"},
    timeout=60.0,
) as response:
    for line in response.iter_lines():
        if line.startswith("data:"):
            try:
                data = json.loads(line[5:].strip())
                node = data.get("node_name", "unknown")
                keys = list(data.get("data", {}).keys())
                print(f"  [{node}] → {keys}")
            except json.JSONDecodeError:
                pass
        elif line.startswith("event:"):
            event_type = line[6:].strip()
            if event_type == "done":
                print("  [完成]")

## 5. 限流测试

API 配置了限流中间件：每分钟最多 30 个请求。

让我们快速发送请求来触发限流。

In [ ]:
# 限流测试 — 快速发送请求
print("=== 限流测试 ===")
print("快速发送 35 个请求...")

results = {200: 0, 429: 0, "other": 0}
for i in range(35):
    r = client.get("/api/v1/graph/stats")
    if r.status_code in results:
        results[r.status_code] += 1
    else:
        results["other"] += 1

print(f"  成功 (200): {results[200]}")
print(f"  被限流 (429): {results[429]}")
print(f"  其他: {results['other']}")

if results[429] > 0:
    print("\n限流正常工作！超过 30 个请求后返回 429 Too Many Requests。")
else:
    print("\n注意: 如果限流未触发，可能是因为请求速度不够快或限流窗口已重置。")

## 6. 错误处理测试

In [ ]:
# 测试空消息（应返回 422）
r = client.post("/api/v1/chat", json={"message": ""})
print(f"空消息 → Status: {r.status_code}")
if r.status_code == 422:
    print(f"  验证错误: {r.json()['detail'][0]['msg']}")

# 测试缺少必填字段（应返回 422）
r = client.post("/api/v1/chat", json={})
print(f"\n缺少 message 字段 → Status: {r.status_code}")
if r.status_code == 422:
    print(f"  验证错误: {r.json()['detail'][0]['msg']}")

## 7. Swagger UI

FastAPI 自动生成交互式 API 文档。

在浏览器中打开以下地址：
- **Swagger UI**: http://localhost:8000/docs
- **ReDoc**: http://localhost:8000/redoc

你可以在 Swagger UI 中直接测试所有端点！

In [ ]:
# 获取 OpenAPI Schema
r = client.get("/openapi.json")
schema = r.json()
print(f"API Title: {schema['info']['title']}")
print(f"API Version: {schema['info']['version']}")
print(f"\n端点列表:")
for path, methods in schema['paths'].items():
    for method, details in methods.items():
        print(f"  {method.upper()} {path} — {details.get('summary', 'N/A')}")

In [ ]:
# 清理
client.close()
print("Client closed. 实验完成！")